In [4]:
!pip install -q kaggle

from google.colab import files
uploaded = files.upload()  # select your kaggle.json here

Saving kaggle.json to kaggle (1).json


In [ ]:
!pip install -q kagglehub
import kagglehub
kagglehub.login()

In [24]:
path = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset")
print("Dataset downloaded to:", path)

import os
os.listdir(path)

100%|██████████| 2.81M/2.81M [00:00<00:00, 104MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/dhivyeshrk/diseases-and-symptoms-dataset/versions/1


['Final_Augmented_dataset_Diseases_and_Symptoms.csv']

In [25]:
import pandas as pd

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
print("Files found:", csv_files)

df = pd.read_csv(os.path.join(path, csv_files[0]))
print(df.shape)
print(df.columns.tolist()[:10])  # confirm which column is the disease label
df.head()

Files found: ['Final_Augmented_dataset_Diseases_and_Symptoms.csv']
(246945, 378)
['diseases', 'anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symptoms', 'sharp chest pain', 'dizziness', 'insomnia', 'abnormal involuntary movements', 'chest tightness']


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [29]:
def normalize_symptom(value):
    import re
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    s = re.sub(r"\s+", "_", s)
    return s

label_col = df.columns[0]  # first column is the disease label in this dataset
SYMPTOM_COLS = [c for c in df.columns if c != label_col]

print("Label column:", label_col)
print(f"{len(SYMPTOM_COLS)} symptom columns")

df[label_col] = df[label_col].astype(str).str.strip()

symptom_vocab = [normalize_symptom(c) for c in SYMPTOM_COLS]
df = df.rename(columns=dict(zip(SYMPTOM_COLS, symptom_vocab)))

Label column: diseases
377 symptom columns


In [31]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

X = df[symptom_vocab].to_numpy(dtype=np.float32)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[label_col])
label_classes = list(label_encoder.classes_)

print(X.shape, y.shape, f"{len(label_classes)} classes")

(246945, 377) (246945,) 773 classes


In [33]:
class_counts = pd.Series(y).value_counts()
print("Classes with fewer than 2 samples:", (class_counts < 2).sum())
print("Classes with fewer than 5 samples:", (class_counts < 5).sum())
print("Smallest classes:\n", class_counts.tail(10))

Classes with fewer than 2 samples: 19
Classes with fewer than 5 samples: 52
Smallest classes:
 178    1
697    1
310    1
282    1
728    1
500    1
465    1
131    1
340    1
406    1
Name: count, dtype: int64


In [38]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

MIN_SAMPLES_PER_CLASS = 2  # need at least 2 for stratified split; a few more gives a less noisy test set

class_counts = pd.Series(y).value_counts()
keep_classes = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index
keep_mask = pd.Series(y).isin(keep_classes).to_numpy()

X_filtered = X[keep_mask]
y_filtered = y[keep_mask]

dropped = len(y) - len(y_filtered)
print(f"Dropped {dropped} rows across {class_counts.size - len(keep_classes)} rare-disease classes")
print(f"Remaining: {X_filtered.shape[0]} rows, {len(keep_classes)} classes")

X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered
)

clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)
print("Trained.")

Dropped 19 rows across 19 rare-disease classes
Remaining: 246926 rows, 754 classes
Trained.


In [37]:
MIN_SAMPLES_PER_CLASS=2
dropped_diseases = [label_encoder.classes_[i] for i in class_counts[class_counts < MIN_SAMPLES_PER_CLASS].index]
print(f"Diseases excluded from v1 (fewer than {MIN_SAMPLES_PER_CLASS} samples):")
for d in sorted(dropped_diseases):
    print(" -", d)

Diseases excluded from v1 (fewer than 2 samples):
 - chronic ulcer
 - diabetes
 - foreign body in the nose
 - gas gangrene
 - heat stroke
 - high blood pressure
 - huntington disease
 - hypergammaglobulinemia
 - kaposi sarcoma
 - myocarditis
 - open wound due to trauma
 - open wound of the cheek
 - open wound of the chest
 - open wound of the head
 - open wound of the knee
 - rocky mountain spotted fever
 - thalassemia
 - turner syndrome
 - typhoid fever


In [40]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print()

present_labels = sorted(set(y_test) | set(y_pred))
present_names = [label_encoder.classes_[i] for i in present_labels]

print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_names))

Accuracy: 0.8161624751953995

                                                          precision    recall  f1-score   support

                               abdominal aortic aneurysm       0.97      1.00      0.98        28
                                        abdominal hernia       0.84      0.96      0.90        81
                                         abscess of nose       0.67      0.91      0.77        58
                                     abscess of the lung       0.67      1.00      0.80         4
                                  abscess of the pharynx       0.74      0.84      0.79        68
                                    acanthosis nigricans       0.67      1.00      0.80         6
                                               acariasis       0.86      0.86      0.86         7
                                               achalasia       0.52      0.82      0.64        17
                                                    acne       0.54      0.83      0.65

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
import json
import joblib
import shutil

os.makedirs("model_artifacts", exist_ok=True)

joblib.dump(clf, "model_artifacts/v1_decision_tree.joblib")

with open("model_artifacts/symptom_vocab.json", "w") as f:
    json.dump(symptom_vocab, f, indent=2)

with open("model_artifacts/label_classes.json", "w") as f:
    json.dump(label_classes, f, indent=2)

description_df.to_csv("model_artifacts/symptom_description.csv", index=False)
precaution_df.to_csv("model_artifacts/symptom_precaution.csv", index=False)

print("Artifacts written:")
!ls -la model_artifacts

In [43]:
shutil.make_archive("v1_model_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v1_model_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>